# 5.1 Class area summary (km² and % cover)

For every Time Interval raster (`TimeInterval*.tif`) and the newest Time Series raster:
count the pixels of each typology class (1-10), then

- **km2** = pixel count × 1016.79 m² / 1,000,000  (31.887054 m cell, ESRI:102033)
- **%cover** = class pixels / total classified pixels × 100

Results are printed per raster and saved to one CSV.

**Packages:** `numpy`, `rasterio` (both in `environment.yml`); `csv`, `time`, `pathlib` are built in.

In [ ]:
### Packages ###
import csv
import time
from pathlib import Path

import numpy as np
import rasterio
from rasterio.windows import Window

In [ ]:
### Parameters ###
TYP = Path(r"./data/5_Typology")
TS_PATH = TYP / "combined_rc_timeseries" / "combined_rc.tif"
OUT_CSV = TYP / "class_area_km2_summary.csv"

PX_M2 = 1016.79      # m2 per pixel
ROWS = 128           # rows read at a time

NAMES = ["Attrition", "Aggregation", "Shrinkage", "Perforation", "Deformation",
         "Persistent", "Enlargement", "Dissection", "Fragmentation per se", "Creation"]

In [ ]:
### Rasters ###
RASTERS = {p.stem: p for p in sorted(TYP.glob("TimeInterval*.tif"))}
RASTERS["TimeSeries 1990-2020"] = TS_PATH

for name, p in RASTERS.items():
    with rasterio.open(p) as src:
        print(name)
        print("   file:", p)
        print("   CRS:", src.crs, "| cell size:", src.res, "| size:", src.width, "x", src.height,
              "| dtype:", src.dtypes[0], "| NoData:", src.nodata)

In [ ]:
### Count pixels per class ###
def count_classes(path):
    """Pixel count for class values 1-10 (NoData and 0 skipped). Returns (counts, pixels with other values)."""
    counts = np.zeros(256, np.int64)
    with rasterio.open(path) as src:
        for r in range(0, src.height, ROWS):
            w = Window(0, r, src.width, min(ROWS, src.height - r))
            a = src.read(1, window=w, masked=True)
            v = np.ma.getdata(a)[~np.ma.getmaskarray(a)].astype(np.int64)
            counts += np.bincount(v[(v >= 0) & (v < 256)], minlength=256)
    return counts[1:11], int(counts.sum() - counts[1:11].sum() - counts[0])


results = {}
for name, p in RASTERS.items():
    t0 = time.time()
    c, other = count_classes(p)
    results[name] = c
    print(f"{name}: {c.sum():,} classified pixels, read in {(time.time() - t0) / 60:.1f} min")
    if other:
        print(f"   WARNING: {other:,} pixels have values that are not 1-10")

In [ ]:
### Print km2 and % cover ###
for name, c in results.items():
    total = c.sum()
    print()
    print(name)
    print(f"{'Class':22}{'pixels':>16}{'km2':>14}{'%cover':>10}")
    for k in range(10):
        print(f"{NAMES[k]:22}{c[k]:>16,}{c[k] * PX_M2 / 1e6:>14,.1f}{100 * c[k] / total:>10.4f}")
    print(f"{'TOTAL':22}{total:>16,}{total * PX_M2 / 1e6:>14,.1f}{100:>10.4f}")

In [ ]:
### Save CSV ###
with open(OUT_CSV, "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["Class"] + [f"{n} {x}" for n in results for x in ("pixels", "km2", "%cover")])
    for k in range(11):
        row = ["TOTAL" if k == 10 else NAMES[k]]
        for n, c in results.items():
            px = int(c.sum()) if k == 10 else int(c[k])
            row += [px, round(px * PX_M2 / 1e6, 1), round(100 * px / c.sum(), 4)]
        w.writerow(row)
print("saved", OUT_CSV)